In [1]:
import os
import pandas as pd
import torch
from datasets import Dataset, DatasetDict
from transformers import (
    RobertaTokenizerFast,
    RobertaForMaskedLM,
    RobertaForQuestionAnswering,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    DefaultDataCollator
)
from sentence_transformers import SentenceTransformer, util

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

TRAIN_FILE = "training_data_en.csv"
EVAL_FILE  = "evaluation_data_en.csv"


c:\Users\kisho\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Using device: cuda


In [2]:
train_df = pd.read_csv(TRAIN_FILE, sep=";", engine="python")
eval_df  = pd.read_csv(EVAL_FILE,  sep=";", engine="python")

corpus = list(train_df["Text"].dropna()) + list(eval_df["Text"].dropna())
print("Corpus size:", len(corpus))

# Save plain text corpus file for MLM
os.makedirs("corpus", exist_ok=True)
with open("corpus/corpus.txt", "w", encoding="utf-8") as f:
    for line in corpus:
        f.write(str(line).replace("\n"," ").strip()+"\n")


Corpus size: 2497


In [3]:
tokenizer = RobertaTokenizerFast.from_pretrained("roberta-base")

mlm_dataset = Dataset.from_dict({"text": corpus})

def tokenize(example):
    return tokenizer(example["text"], truncation=True, padding="max_length", max_length=128)

tokenized_mlm = mlm_dataset.map(tokenize, batched=True, remove_columns=["text"])

data_collator_mlm = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15
)


Map: 100%|██████████| 2497/2497 [00:00<00:00, 12996.64 examples/s]


In [4]:
mlm_model = RobertaForMaskedLM.from_pretrained("roberta-base").to(device)

mlm_args = TrainingArguments(
    output_dir="roberta-fin-mlm",
    num_train_epochs=100,       # You can push 20 later for better results
    per_device_train_batch_size=16,
    save_steps=1000,
    logging_steps=200,
    learning_rate=5e-5,
)

mlm_trainer = Trainer(
    model=mlm_model,
    args=mlm_args,
    train_dataset=tokenized_mlm,
    data_collator=data_collator_mlm,
)

print("Starting Domain Adaptation Pretraining...")
mlm_trainer.train()
mlm_trainer.save_model("roberta-fin-mlm")
tokenizer.save_pretrained("roberta-fin-mlm")
print("DAPT Complete ✔ Model saved.")


Starting Domain Adaptation Pretraining...


Step,Training Loss
200,1.762100
400,1.579600
600,1.506700
800,1.425600
1000,1.352800
1200,1.324200
1400,1.283000
1600,1.204200
1800,1.142900
2000,1.107000


DAPT Complete ✔ Model saved.


In [5]:
def load_qa_data(file):
    df = pd.read_csv(file, sep=";", engine="python")
    data=[]
    for _,row in df.dropna(subset=["Text","Question"]).iterrows():
        if "Answer" in df.columns and isinstance(row["Answer"],str):
            ans=row["Answer"].strip()
            pos=row["Text"].find(ans)
            if pos==-1: continue
            data.append({
                "id":str(row["ID"]),
                "context":row["Text"],
                "question":row["Question"],
                "answers":{"text":[ans],"answer_start":[pos]}
            })
        else:
            data.append({
                "id":str(row["ID"]),
                "context":row["Text"],
                "question":row["Question"],
                "answers":{"text":[],"answer_start":[]}
            })
    return Dataset.from_list(data)

train_qa = load_qa_data(TRAIN_FILE)
eval_qa  = load_qa_data(EVAL_FILE)

split=train_qa.train_test_split(0.2,seed=42)
qa_dataset = DatasetDict({"train":split["train"],"validation":split["test"]})


In [6]:
from transformers import AutoTokenizer
tokenizer = RobertaTokenizerFast.from_pretrained("roberta-fin-mlm")

MAX_LEN=384
STRIDE=128

def preprocess_qa(examples):
    inputs = tokenizer(
        examples["question"],
        examples["context"],
        truncation="only_second",
        max_length=MAX_LEN,
        stride=STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length"
    )

    offset_mapping = inputs.pop("offset_mapping")
    sample_map = inputs.pop("overflow_to_sample_mapping")

    start_positions=[]
    end_positions=[]

    for i, offsets in enumerate(offset_mapping):
        sample_idx = sample_map[i]
        answers = examples["answers"][sample_idx]
        
        input_ids = inputs["input_ids"][i]
        cls_index = input_ids.index(tokenizer.cls_token_id)

        if len(answers["answer_start"])==0:
            start_positions.append(cls_index)
            end_positions.append(cls_index)
            continue

        start_char = answers["answer_start"][0]
        end_char   = start_char + len(answers["text"][0])

        seq_ids = inputs.sequence_ids(i)

        # find context token span
        context_start=None
        context_end=None
        
        for idx,s in enumerate(seq_ids):
            if s==1 and context_start is None: context_start=idx
            if s==1: context_end=idx

        # answer not in span
        if offsets[context_start][0] > start_char or offsets[context_end][1] < end_char:
            start_positions.append(cls_index)
            end_positions.append(cls_index)
        else:
            # locate token positions
            start_idx=context_start
            while start_idx<=context_end and offsets[start_idx][0]<=start_char:
                start_idx+=1
            end_idx=context_end
            while end_idx>=context_start and offsets[end_idx][1]>=end_char:
                end_idx-=1

            start_positions.append(start_idx-1)
            end_positions.append(end_idx+1)

    inputs["start_positions"]=start_positions
    inputs["end_positions"]=end_positions
    return inputs


processed = qa_dataset.map(
    preprocess_qa,
    batched=True,
    remove_columns=qa_dataset["train"].column_names   # IMPORTANT!
)


Map: 100%|██████████| 398/398 [00:00<00:00, 990.98 examples/s] 


In [ ]:
qa_model = RobertaForQuestionAnswering.from_pretrained("roberta-fin-mlm").to(device)

BATCH_SIZE = 2                 # lower = faster & less VRAM use
MAX_LEN = 256                  # faster than 384
EPOCHS = 10                     # train faster first, tune later

args = TrainingArguments(
    output_dir="fin-qa-dapt",
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    learning_rate=3e-5,
    logging_steps=50,
    save_steps=300,
    fp16=True,                  # 🔥 HUGE speed boost on GPU
    gradient_checkpointing=True, # reduce memory usage further
    do_train=True,
    do_eval=True
)

trainer = Trainer(
    model=qa_model,
    args=args,
    train_dataset=processed["train"],
    eval_dataset=processed["validation"],
    tokenizer=tokenizer,
    data_collator=DefaultDataCollator()
)

print("🚀 Starting FAST QA fine-tuning on GPU...")
trainer.train()
print("📊 Evaluation:", trainer.evaluate())


Some weights of RobertaForQuestionAnswering were not initialized from the model checkpoint at roberta-fin-mlm and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\kisho\AppData\Local\Temp\ipykernel_8928\1214926664.py:21: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


🚀 Starting FAST QA fine-tuning on GPU...


Step,Training Loss
50,3.963700
100,1.772600
150,1.294000
200,0.824600
250,0.791500
300,0.892500
350,0.677700
400,0.564600
450,0.506900
500,0.465600


📊 Evaluation: {'eval_loss': 1.0804108381271362, 'eval_runtime': 4.1325, 'eval_samples_per_second': 96.309, 'eval_steps_per_second': 48.155, 'epoch': 10.0}


In [8]:
trainer.save_model("fin-qa-dapt")
tokenizer.save_pretrained("fin-qa-dapt")
print("\n🎉 Model saved successfully at fin-qa-dapt/")


🎉 Model saved successfully at fin-qa-dapt/


In [9]:
sas_model=SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

def predict(context,question):
    inputs=tokenizer(question,context,return_tensors="pt",truncation=True,max_length=384).to(device)
    with torch.no_grad():
        out=qa_model(**inputs)
    s=int(out.start_logits.argmax())
    e=int(out.end_logits.argmax())
    if e<s: return ""
    ans=inputs.input_ids[0][s:e+1]
    return tokenizer.decode(ans,skip_special_tokens=True)

pred=[];truth=[]
for row in qa_dataset["validation"]:
    pred.append(predict(row["context"],row["question"]))
    truth.append(row["answers"]["text"][0])

EM=sum([p.strip()==g.strip() for p,g in zip(pred,truth)])/len(truth)
SAS=float(torch.diag(util.cos_sim(
    sas_model.encode(pred,convert_to_tensor=True),
    sas_model.encode(truth,convert_to_tensor=True)
)).mean())

print(f"📊 Validation Results:\nEM={EM*100:.2f}%\nSAS={SAS:.4f}")


📊 Validation Results:
EM=82.41%
SAS=0.9528


In [10]:
final=[]
for row in eval_qa:
    final.append(predict(row["context"],row["question"]))

pd.DataFrame({"ID":eval_qa["id"],"Answer":final}).to_csv("submission.csv",sep=";",index=False)
print("Saved submission.csv ✔")


Saved submission.csv ✔
